# 📘 Notebook – Etapa Descritiva (base para Análise Preditiva)

## Análise Exploratória e Qualidade dos Dados (EDA)

Esta seção descreve a estrutura e a qualidade do conjunto de dados utilizados no estudo.  
**Fonte:** [ Lesões no Futebol Europeu 2020–2025 (Kaggle) ]( https://www.kaggle.com/datasets/sananmuzaffarov/european-football-injuries-2020-2025 )  
 **Licença:** CC BY-SA 4.0 — https://creativecommons.org/licenses/by-sa/4.0/

---

### 1) Importações e carregamento do conjunto de dados

In [ ]:
import pandas as pd
import numpy as np
import sys

## 2) Visão geral

### Coleta de Dados do Kaggle

In [ ]:
# Acessando dataset European Football Injuries (2020-2025)
%cd /content/pmv-si-2026-1-pe7-t1-g3-analytics_lesao_futebol
sys.path.append('/content/pmv-si-2026-1-pe7-t1-g3-analytics_lesao_futebol')

In [ ]:
from src.data_loader import load_injuries_data

df_injuries = load_injuries_data()

In [ ]:

print ("Linhas:", df_injuries.shape[0])
print ("Colunas:", df_injuries.shape[1])
print ("\nNome das colunas:", list(df_injuries.columns))

In [ ]:
print(df_injuries.dtypes)

In [ ]:
df_injuries.head()

In [ ]:
df_injuries.tail()

O dataset contém registros de eventos de lesão/indisponibilidade de jogadores profissionais
de futebol nas cinco principais ligas europeias: **Bundesliga, Premier League, La Liga,
Ligue 1 e Série A**. Cada linha representa um evento único de afastamento associado a um
jogador, com informações de contexto (clube e liga), características do atleta (idade e
posição) e impacto esportivo (dias afastados e jogos perdidos).
  - **
Total de registros:** 15.603 - **   
Total de atributos: ** 11 - ** Temporados presentes   
: ** 20/21 , 21/22 , 22/23 , 23/24 , 24/25

In [ ]:
dicionario = pd.DataFrame({
    "Atributo": [
        "Season" , "Injury" , "Days" , "Games Missed" , "injury_from_parsed" ,
        "injury_until_parsed" , "player_name" , "player_age" , "player_position" ,
        "club" , "league"
    ],
    "Descricao": [
        "Temporada da ocorrência" ,
         "Tipo/descrição da lesão ou condição" ,
         "Duração do afastamento (texto, ex: '43 dias')" ,
         "Número de partidas perdidas" ,
         "Data de início do afastamento" ,
         "Data de fim do afastamento" ,
         "Nome do jogador" ,
         "Idade do jogador no evento" ,
         "Posição tática do jogador" ,
         "Clube do jogador" ,
         "Liga associada ao clube"
    ],
    "Tipo (arquivo)": [
        "texto" , "texto" , "texto" , "inteiro" , "texto (dados)" ,
        "texto (dados)" , "texto" , "inteiro" , "texto" , "texto" ,
        "texto"
    ],
    "Unidade/Formato": [
        "—" , "—" , "dias" , "jogos" , "M/D/YYYY" , "M/D/YYYY" ,
        "—" , "anos" , "—" , "—" , "—"
    ],
    "Exemplos": [
        "20/21, 24/25",
        "Lesão no tendão da coxa, lesão no joelho",
        "43 dias, 8 dias" ,
         "9, 2, 145" ,
         "28/01/2021" ,
         "11/03/2021" ,
         "Benjamin Pavard" ,
         "19, 25, 43" ,
         "Goleiro, Zagueiro Central" ,
         "Bayern de Munique" ,
          "Bundesliga, Seria A"
    ]
})

print(dicionario.to_string(index=False))

In [ ]:
df_injuries = df_injuries.rename(columns={
    "Season": "temporada",
    "Injury": "tipo_lesao",
    "Days": "duracao_dias",
    "Games missed": "jogos_perdidos",
    "injury_from_parsed": "data_inicio_lesao",
    "injury_until_parsed": "data_fim_lesao",
    "player_name": "jogador",
    "player_age": "idade",
    "player_position": "posicao",
    "club": "clube",
    "league": "liga",

})

Para uma anális mais didática, realizamos a tradução das colunas do dataset para pt_BR

In [ ]:
# Lista quantidade de linhas e coculas

print(f'''
    Colunas: {len(df_injuries.columns)} - {df_injuries.columns.values}
  ''')

## 3) Dados ausentes

In [ ]:
missing = df_injuries.isna().sum()
print("\nMissing por coluna:")
print(missing)

Não foram encontrados valores ausentes em nenhuma das 11 colunas do conjunto de dados
(** 0 % de dados faltantes**). O conjunto de dados está completamente preenchido, o que
elimina a necessidade de imputação ou remoção de registros nesta etapa.

## 4) Duplicatas


In [ ]:
dup_count = df_injuries.duplicated().sum()
print("\nDuplicatas exatas (linhas idênticas):", dup_count)

Não foram identificadas duplicatas exatas de linha (** 0 registros duplicados**).
Cada observação representa um evento de afastamento distinto, garantindo a
integridade dos registros para análises longitudinais.

### 5) Consistência temporal e coerência entre variáveis

In [ ]:
# Converter dias para numérico
days_num = df_injuries['duracao_dias'].astype(str).str.extract(r"(\d+)")[0].astype(int)
print("\nDura das lesões (dias) - min/max:", np.nanmin(days_num), "e", np.nanmax(days_num))

# Datas
from_dt = pd.to_datetime(df_injuries['data_inicio_lesao'], errors="coerce")
until_dt = pd.to_datetime(df_injuries['data_fim_lesao'], errors="coerce")

print("\nFalhas de parsing - data_inicio_lesao:", from_dt.isna().sum())
print("\nFalhas de parsing - data_fim_lesao:", until_dt.isna().sum())

print("\nPeríodo (datas):")
print("Min data_inicio_lesao:", from_dt.min())
print("Max data_fim_lesao:", until_dt.max())

# Inconsistência: fim antes do início
until_before_from = ((~from_dt.isna()) & (~until_dt.isna()) & (until_dt < from_dt)).sum()
print("\nCasos com until < from:", until_before_from)

# Coerência: Days vs (until - from)
duration_from_dates = (until_dt - from_dt).dt.days
mask_valid = (~days_num.isna()) & (~duration_from_dates.isna())
diff = (days_num[mask_valid] - duration_from_dates[mask_valid]).abs()

mismatch_gt3 = (diff > 3).sum()
print("\nDivergências | dias - (until-from)| > 3 dias:", mismatch_gt3)

A análise de consistência temporal revelou:

- **Análise de dados:** sem falhas de conversão em ambas as colunas de dados.
- **Período coberto:** de ** 2020 -02-01** a ** 2026 -01- 15 ** *(o limite superior ultrapassa
   2025 em função de dados de retorno registrados no arquivo)*.
- **Ordem temporal:** nenhum caso com dados de fim anterior à data de   início
  (** sem exceções**).
- **Coerência `Dias` vs intervalo entre dados:** nenhuma divergência absoluta superior
  a 3 dias (** sem ocorrências**), diminuição boa consistência interna entre a duração
  declarada e como dados registrados.

## 6) Convertendo dados da tabela

In [ ]:
# Converter dias para numérico
df_injuries['duracao_dias'] = df_injuries['duracao_dias'].astype(str).str.extract(r"(\d+)")[0].astype(int)
print("\nDura das lesões (dias) - min/max:", np.nanmin(days_num), "e", np.nanmax(days_num))

# Datas
df_injuries['data_inicio_lesao'] = pd.to_datetime(df_injuries['data_inicio_lesao'], errors="coerce")
df_injuries['data_fim_lesao'] = pd.to_datetime(df_injuries['data_fim_lesao'], errors="coerce")

print(df_injuries.dtypes)

## 7) Outliers via IQR

In [ ]:
def iqr_outliers(serie):
  s = pd.to_numeric(serie, errors="coerce").dropna()
  q1 = s.quantile(.25)
  q3 = s.quantile(.75)
  iqr = q3 - q1
  low = q1 - 1.5 * iqr
  high = q3 + 1.5 * iqr
  out_count = ((s < low) | (s > high)).sum()
  return int(out_count), float(low), float(high)


out_games_cnt, out_games_low, out_games_high = iqr_outliers(df_injuries['jogos_perdidos'])
out_days_cnt, out_days_low, out_days_high = iqr_outliers(df_injuries['duracao_dias'])
print("\nOutliers (IQR) - Jogos perdidos:", out_games_cnt, "| limites", (out_games_low, out_games_high))
print("\nOutliers (IQR) - Dias (num):", out_days_cnt, "| limites", (out_days_low, out_days_high))

Para identificar valores extremos utilizados-se o seletivos do **IQR (1,5×IQR)**. Os
outliers foram **contabilizados, mas não removidos**, pois podem representar casos
reais de lesões graves (ex.: ruptura do ligamento cruzado anterior) e são relevantes para análises de severidade.

| Variável | Faixa observada | Limite superior IQR | Outliers identificados |
|---|---|---|---|
| `Jogos perdidos` | 1 a 145 jogos | ~13,5 jogos | **1.367 registros** |
| `Dias` (numérico) | 1 a 1.013 dias | ~84 dias | **1.497 registros** |

**Implicação analítica:** devido à presença de caudas mais longas, recomenda-se:
- Utilização **mediana e percentis** em estatísticas descritivas.
- Avaliar **winsorização** ou **segmentação por tipo de lesão/posição** em modelos
  preditivos, conforme o objetivo do estudo.

## 7) Síntese da qualidade dos dados

In [ ]:
resumo = pd.DataFrame({
    "Dimensão": [
        "Valores faltantes",
        "Duplicatas exatas",
        "Falhas de parsing (datas)",
        "Inconsistências temporais (até <de)",
        "Divergências Dias vs intervalo de dados",
        "Valores atípicos — Jogos perdidos (intervalo interquartil)",
        "Valores atípicos — Dias numéricos (IQR)"
    ],
    "Resultao": [
        "0 em todas as colunas (0% faltando)",
        "0 linhas duplicadas",
        "0 falhas em ambas as colunas",
        "0",
        "0 entradas > 3 dias",
        "1.367 registros (máx. 145 jogos)",
        "1.497 registros (máx. 1.013 dias)"
    ],
    "Avaliação": [
        "✅ Completo",
        "✅ Sem duplicatas",
        "✅ Dados válidos",
        "✅ Consistente",
        "✅ Coerente",
        "⚠️ Caudas longas — manter e monitorar",
        "⚠️ Caudas longas — manter e monitorar"
    ]
})

print(resumo.to_string(index=False))

O conjunto de dados apresenta **alta qualidade estrutural**: sem falta, sem duplicatas e com
consistência interna entre variáveis. O único ponto de atenção são as caudas mais longas
nas variáveis ​​de duração e impacto, esperadas em dados de lesões esportivas e que
Devem ser consideradas nas escolhas metodológicas das etapas seguintes.